In [2]:
import os
import re
import sys

import itertools 
import scipy
import matplotlib
import glob
%matplotlib inline
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import scipy.sparse as sp_sparse
import scipy.io as sio
import matplotlib.mlab as mlab

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from multiprocessing import Pool
from collections import defaultdict
from scipy import sparse
from scipy.sparse import csr_matrix
from scipy.cluster.hierarchy import dendrogram, linkage
import scipy.io as io

In [3]:
global_df = pd.read_csv('/project/GCRB/Hon_lab/s426305/Analysis/IGVF/20240219_WTC11_TFPerturb_CM-full_all/pySpade/Manhattan_plots/filtered_df.csv')

In [4]:
global_df.shape

(27578, 18)

In [5]:
#Load annotation file 
annot_dict_file = '/project/GCRB/Hon_lab/s426305/Analysis/IGVF/20240219_WTC11_TFPerturb_CM-full_all/pySpade/annotation_dict_hg38.txt'
annot_dict = {}
with open(annot_dict_file) as f:
    for line in f:
        region_id, annotation = line.strip().split("\t")
        annot_dict.update({region_id : annotation})

In [6]:
TF_region_df = pd.DataFrame.from_dict(annot_dict, columns=['TF'], orient='index')

In [7]:
TF_global_df = global_df[global_df['gene_names'].isin(TF_region_df['TF'])] 

In [8]:
TF_global_df.shape

(3010, 18)

In [10]:
rank_TF_list = [annot_dict[i[0]] for i in collections.Counter(TF_global_df['region']).most_common()]

###  RE df

In [11]:
df_column_list = [
        'idx', 'gene_names', 'chromosome', 'pos', 'strand', 
        'color_idx', 'chr_idx', 
        'region', 'num_cell', 'bin',
        'log(pval)-hypergeom', 'fc', 'Significance_score', 'fc_by_rand_dist_cpm', 'pval-empirical', 'cpm_perturb', 'cpm_bg']

In [12]:
GLBOAL_HITS = '/project/GCRB/Hon_lab/s426305/Analysis/IGVF/20241209_WTC11_lenti_TFPerturb_RE/pySpade/unfiltered_global_df.csv'

In [13]:
FILE_DIR = '/project/GCRB/Hon_lab/s426305/Analysis/IGVF/20241209_WTC11_lenti_TFPerturb_RE/pySpade/'

In [14]:
#Load unfilter global hits df 
global_df = pd.read_csv(GLBOAL_HITS)[df_column_list]

In [15]:
CUTOFF_EXP = 0.05
CUTOFF_FC = 0.1
CUTOFF_SIG = -10

In [16]:
#Load expressed genes 
gene_seq = np.load(FILE_DIR + 'Trans_genome_seq.npy', allow_pickle=True)
express_level = np.load(FILE_DIR + 'Perc_cell_expr.npy')
express_idx = np.where(express_level > CUTOFF_EXP)[0]

In [17]:
#filter df 
express_df = global_df[global_df['gene_names'].isin(gene_seq[express_idx])\
                  &((global_df['fc_by_rand_dist_cpm'] > (1+CUTOFF_FC)) | (global_df['fc_by_rand_dist_cpm'] < (1-CUTOFF_FC)))\
                  &(global_df['Significance_score'] < CUTOFF_SIG)\
                  &(global_df['pval-empirical'] < 0.001)
                  &(global_df['log(pval)-hypergeom'] < -3)]

In [30]:
express_df['TF_annotation'] = [annot_dict[i] for i in express_df['region']] 
TF_global_df['TF_annotation'] = [annot_dict[i] for i in TF_global_df['region']] 

/tmp/ipykernel_62792/3074219906.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  express_df['TF_annotation'] = [annot_dict[i] for i in express_df['region']]


In [19]:
RE_annotation_df = pd.read_csv('/project/GCRB/Hon_lab/s159317/IGVF/scTE_mapping/CM.TF-Perturb-Seq/temp/rmsk.txt', sep='\t', 
                           names=['1', '2', '3', '4', '5', 'chr', 'start', 'end', 'pos', 'strand', 'name', 'cate', 'names', '6', '7', '8', '9'])

In [20]:
RE_annotation_df.head()

,1,2,3,4,5,chr,start,end,pos,strand,name,cate,names,6,7,8,9
0,585,463,13,6,17,chr1,10000,10468,-248945954,+,(TAACCC)n,Simple_repeat,Simple_repeat,1,471,0,1
1,585,3612,114,215,13,chr1,10468,11447,-248944975,-,TAR1,Satellite,telo,-399,1712,483,2
2,585,484,251,132,0,chr1,11504,11675,-248944747,-,L1MC5a,LINE,L1,-2382,395,199,3
3,585,239,294,19,10,chr1,11677,11780,-248944642,-,MER5B,DNA,hAT-Charlie,-74,104,1,4
4,585,318,230,37,0,chr1,15264,15355,-248941067,-,MIR3,SINE,MIR,-119,143,49,5


In [21]:
subset_express_df = express_df[(express_df['gene_names'].isin(RE_annotation_df['name']))]

In [34]:
np.unique(subset_express_df["TF_annotation"]).shape

(423,)

In [22]:
RE_TF_list = [i[0] for i in collections.Counter(subset_express_df['TF_annotation']).most_common()]

In [31]:
for i in RE_TF_list:
    TF_common = set(TF_global_df[TF_global_df['TF_annotation'] == i]['gene_names']).intersection(set(RE_TF_list))
    print(i + '\t' + str(TF_common))

TCF7	{'TSHZ2', 'IRX3', 'MYOCD', 'CREM', 'ESRRG'}
ZNF630	set()
DMRTA1	set()
ESRRG	{'ESRRG'}
NR1H3	set()
FOXG1	set()
HELT	set()
ZBTB21	{'GATA5'}
CPEB1	set()
ZMAT2	set()
GLI3	{'HAND1', 'GLI3', 'FOXK1'}
ZNF449	set()
LHX1	set()
ZFP64	{'ZFP64', 'HAND1', 'HOXB4', 'HOXB2', 'KMT2A', 'ESRRG', 'MSX2'}
TBPL1	set()
ZNF276	{'ZNF276'}
SPIC	set()
CSRNP3	{'CSRNP3'}
FOXE1	set()
LCORL	{'LCORL'}
SP110	{'MYOCD'}
BAZ1B	{'IRX3', 'BAZ1B', 'TCERG1'}
MYT1	set()
HIRA	set()
ZNF202	{'ZNF202'}
ZNF30	{'ZNF30'}
REXO4	{'REXO4'}
ZNF805	set()
ZNF532	{'EGR1', 'ZNF141', 'ESRRG', 'ZNF532'}
BSX	set()
ZNF649	{'ZNF649'}
ZNF628	set()
GAS7	{'GAS7'}
IKZF2	{'IKZF2'}
CHAMP1	{'CREB5', 'JUN', 'TBX18', 'CHAMP1', 'ESRRG'}
RFX7	{'SMAD7'}
KMT2A	{'SOX6', 'HOXB4', 'TBX18', 'KMT2A', 'ESRRG'}
ZNF880	{'ZNF880'}
LZTS1	set()
OR2T5	set()
CENPB	{'CENPB'}
HES6	set()
TCEB2	set()
FUBP3	{'FUBP3'}
ZNF431	{'ZNF431'}
ZNF561	{'ZNF561'}
HOXC6	set()
ZNF679	set()
ZNF562	set()
TADA2B	{'ZNF385A', 'EEA1', 'NR4A3', 'TBX18', 'SOX6', 'PHTF1', 'MYOCD', 'NCOA6', '

In [35]:
subset_express_df

,idx,gene_names,chromosome,pos,strand,color_idx,chr_idx,region,num_cell,bin,log(pval)-hypergeom,fc,Significance_score,fc_by_rand_dist_cpm,pval-empirical,cpm_perturb,cpm_bg,TF_annotation
128561,62842,MER34A1,NaN,NaN,+,0,24,chr14:35403160-35403255,788,800,-10.402540,2.150119,-10.139871,2.133280,0.0,13.600745,6.370198,NFKBIA
234932,67935,AluSq,NaN,NaN,+,0,24,chr15:82647585-82648021,850,800,-10.200421,1.181449,-10.711242,1.180137,0.0,332.989151,282.159954,CPEB1
234939,79974,AluSx4,NaN,NaN,+,0,24,chr15:82647585-82648021,850,800,-11.814065,1.235187,-11.745783,1.232939,0.0,185.153446,150.170547,CPEB1
238945,80427,Kanga1a,NaN,NaN,+,0,24,chr15:82647585-82648021,850,800,-9.228898,1.394545,-10.311442,1.402169,0.0,17.230537,12.285619,CPEB1
238949,73752,L1M2,NaN,NaN,+,0,24,chr15:82647585-82648021,850,800,-10.671792,1.156556,-10.940925,1.155351,0.0,268.157090,232.098763,CPEB1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45020253,79314,Ricksha,NaN,NaN,+,0,24,chr5:178941224-178941469,572,600,-9.093342,2.468002,-10.017832,2.486340,0.0,6.778835,2.720453,ZNF454
45188299,81638,HERVL-int,NaN,NaN,+,0,24,chr2:70790394-70790673,562,600,-9.168361,1.337419,-10.665309,1.342354,0.0,53.180660,39.614912,FIGLA
45404080,81824,LTR57-int,NaN,NaN,+,0,24,chr19:12365551-12365666,588,600,-8.606282,3.110194,-10.093491,3.099913,0.0,6.296399,2.024379,ZNF442
45447788,80555,LTR91,NaN,NaN,+,0,24,chr8:124973307-124973501,662,600,-8.882677,4.109250,-12.029823,4.234418,0.0,5.592406,1.313064,ZNF572
